# Context and Sessions

Every run so far starts fresh. The agent forgets what you said before. A **session** fixes
that. It saves a ninja's chat history between runs. There are three kinds:
`SQLiteSession` (quick), `SQLAlchemySession` (works with any real database), and
`EncryptedSession` (scrambles the saved chat so nobody can read it).
`RunContextWrapper` is a _different_ idea: it is a value you read inside a function. It is not a saved chat.

Needs `openai-agents[sqlalchemy,encrypt]`, plus `aiosqlite` and `greenlet`:

```bash
pip install "openai-agents[sqlalchemy,encrypt]==0.22.3" aiosqlite greenlet
```


## Remember turns with `SQLiteSession`

Pass `session=` to `Runner.run`. The SDK loads the old chat before the run and saves the new
messages after it. `db_path=":memory:"` keeps the chat only while the program runs. A file
path keeps it even after a restart.


In [1]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
)

# Initialize the agent
leaf_assistant = Agent(
    name="Leaf Village Assistant",
    instructions="test",
)

In [2]:
from agents import (
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.memory import (
    SQLiteSession,  # a session that saves chat history in a SQLite database (or in memory)
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
)


# Demo: two questions on one session; question 2 can see question 1's chat
async def demo_sqlite_session() -> None:
    # Initialize the session
    session = SQLiteSession(
        session_id="ninja-naruto-uzumaki",
        db_path=":memory:",
    )

    # Script the model: one reply for each of the two questions
    model = ScriptedModel(
        [
            [assistant_message("Naruto Uzumaki is Kage-level.")],
            [assistant_message("Team 7.")],
        ]
    )

    for question in ["Who is Naruto Uzumaki?", "What's his team?"]:
        # Run the Agent
        await Runner.run(
            leaf_assistant,
            question,
            session=session,
            run_config=RunConfig(
                model=model,
                workflow_name="e11-sqlite",
            ),
        )

    # proves session= carried question 1 forward: the model saw 3 items on turn 2 but only 1 on turn 1
    print(f"[history items] turn 1: {len(model.calls[0].input)}, turn 2: {len(model.calls[1].input)}")


await demo_sqlite_session()
# Outside a notebook: asyncio.run(demo_sqlite_session())

[history items] turn 1: 1, turn 2: 3


## Use a real database and lock it with `SQLAlchemySession` and `EncryptedSession`

It is the same session idea, saved in a new place. `SQLAlchemySession.from_url(...)` works with
any SQLAlchemy database. `create_tables=True` builds the tables for you. (In a real app, you
manage table changes yourself.) `EncryptedSession` wraps any other session and scrambles
what it saves with a secret key. When you read through it, you get normal text back. But the
session underneath only holds scrambled text.


In [3]:
from agents import (
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.extensions.memory.encrypt_session import (
    EncryptedSession,  # wraps any session and locks its data with a secret key
)
from agents.extensions.memory.sqlalchemy_session import (
    SQLAlchemySession,  # a session that saves chat history in any SQLAlchemy database
)
from agents.memory import (
    SQLiteSession,  # a session that saves chat history in a SQLite database (or in memory)
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
)


# Demo: save one turn in a SQLAlchemy database, then in an encrypted session, and read both back
async def demo_sqlalchemy_and_encrypted() -> None:
    # Initialize the session
    sa_session = SQLAlchemySession.from_url(
        "ninja-sa-demo",
        # url: any connection string that SQLAlchemy understands
        url="sqlite+aiosqlite:///:memory:",
        # create_tables: makes the tables for you
        create_tables=True,
    )

    # Initialize the session
    plain_session = SQLiteSession(
        session_id="ninja-team-secrets",
        db_path=":memory:",
    )
    encrypted_session = EncryptedSession(
        # session_id: the name of this chat history
        session_id="ninja-team-secrets",
        # underlying_session: where the locked items are really stored
        underlying_session=plain_session,
        # encryption_key: use a real secret in production, not text in the code
        encryption_key="a-very-secret-passphrase-for-testing",
    )

    for session in [sa_session, encrypted_session]:
        # Script the model
        model = ScriptedModel([[assistant_message("Team 7 details.")]])
        # Run the Agent
        await Runner.run(
            leaf_assistant,
            "Secret team info",
            # session: saves and loads this chat's history
            session=session,
            # run_config: settings for just this run
            run_config=RunConfig(model=model, workflow_name="e11-store"),
        )

    # proves the SQLAlchemy store really saved the turn's items
    print("[sqlalchemy items stored]", len(await sa_session.get_items()))
    # proves get_items() unlocks the data for the caller
    print("[via EncryptedSession, decrypted]", (await encrypted_session.get_items())[0])
    # proves the data underneath is really scrambled, not just hidden
    print("[underlying session, scrambled text]", str((await plain_session.get_items())[0])[:100], "...")


await demo_sqlalchemy_and_encrypted()
# Outside a notebook: asyncio.run(demo_sqlalchemy_and_encrypted())

[sqlalchemy items stored] 2
[via EncryptedSession, decrypted] {'content': 'Secret team info', 'role': 'user'}
[underlying session, scrambled text] {'__enc__': 1, 'v': 1, 'kid': 'hkdf-v1', 'payload': 'gAAAAABqw-PbJ6R15I44IlIxa_CxcQpQ-LjNYuI7pNqPRtH ...


## Shrink long history with `OpenAIResponsesCompactionSession`

A long chat costs more and more. This session wraps another session. After a turn, once the saved
chat gets long, it asks OpenAI to squeeze the old messages into a shorter version
(`responses.compact`). That call goes over the internet, so the offline cell below only builds the
session. The live cell at the end really triggers it.


In [4]:
from agents.memory import (
    OpenAIResponsesCompactionSession,  # a session that shrinks old chat history when it gets long
    SQLiteSession,  # a session that saves chat history in a SQLite database (or in memory)
)

# Initialize the session
compaction_session = OpenAIResponsesCompactionSession(
    # session_id: the name of this chat history
    session_id="ninja-compacted",
    # underlying_session: where the items really live. Compaction shortens them on top of it
    underlying_session=SQLiteSession(session_id="ninja-compacted", db_path=":memory:"),
    # model: which model writes the shorter summary
    model="gpt-5.6-luna",
)
print("[constructed]", type(compaction_session).__name__)
print("[note] actually triggering compaction needs a live OPENAI_API_KEY, see the live cell below")

[constructed] OpenAIResponsesCompactionSession
[note] actually triggering compaction needs a live OPENAI_API_KEY, see the live cell below


## Run it live — optional, needs a real key

Every session type gets the same real test with two turns: tell the agent a name, then ask for the
name back. This runs on `SQLiteSession`, `SQLAlchemySession`, and `EncryptedSession`. For the
encrypted one, the cell also looks at the session underneath to show the saved turns are scrambled.

The last test is live-only: `OpenAIResponsesCompactionSession`. Its `should_trigger_compaction`
rule is set to squeeze at 2 items instead of 10, so a real squeeze happens after turn 2. Without
squeezing, the session would hold 4 items after turn 2. A smaller number shows the history was
replaced by a shorter one. Then a third turn still remembers the name. Whether the model
remembers is up to the model, so the cell prints its answer. Skipped when there is no `OPENAI_API_KEY`.


In [5]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.memory import (
    Session,  # the shape every session type must follow
)


# Live helper: asks two real questions on any session type (say a name, then ask for it back)
async def recall_across_turns(agent: Agent, session: Session, label: str) -> None:
    final_answer = ""
    for message in ["My name is Shikamaru Nara, remember that.", "What's my name?"]:
        # Run the Agent
        result = await Runner.run(
            agent,
            message,
            session=session,
            run_config=RunConfig(workflow_name=f"e11-live-{label}"),
        )
        final_answer = result.final_output
    # proves session= carried turn 1 into turn 2 with a real model, with no extra history setup
    print(f"[live {label}] {len(await session.get_items())} items stored, turn 2 answer: {final_answer}")

## Test memory on every session type


In [6]:
import os  # reads OPENAI_API_KEY from the environment

from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
)
from agents.extensions.memory.encrypt_session import (
    EncryptedSession,  # wraps any session and locks its data with a secret key
)
from agents.extensions.memory.sqlalchemy_session import (
    SQLAlchemySession,  # a session that saves chat history in any SQLAlchemy database
)
from agents.memory import (
    SQLiteSession,  # a session that saves chat history in a SQLite database (or in memory)
)
from dotenv import load_dotenv  # loads the .env file into the environment

load_dotenv()


# Demo: a real two-turn memory test on SQLite, SQLAlchemy, and the encrypted session
async def demo_live_memory() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    # Initialize the agent
    agent = Agent(
        name="Leaf Village Assistant",
        instructions="Answer briefly.",
    )

    # Initialize the session
    sqlite_session = SQLiteSession(
        session_id="ninja-live-sqlite",
        db_path=":memory:",
    )
    await recall_across_turns(agent, sqlite_session, "sqlite")

    # Initialize the session
    sa_session = SQLAlchemySession.from_url(
        "ninja-live-sa",
        url="sqlite+aiosqlite:///:memory:",
        create_tables=True,
    )
    await recall_across_turns(agent, sa_session, "sqlalchemy")

    # Initialize the session
    plain_session = SQLiteSession(
        session_id="ninja-live-encrypted",
        db_path=":memory:",
    )
    encrypted_session = EncryptedSession(
        # session_id: the name of this chat history
        session_id="ninja-live-encrypted",
        # underlying_session: where the locked items are really stored
        underlying_session=plain_session,
        # encryption_key: use a real secret in production, not text in the code
        encryption_key="a-very-secret-passphrase-for-testing",
    )
    await recall_across_turns(agent, encrypted_session, "encrypted")
    # proves the real turns are saved as scrambled text, not readable text
    print("[live encrypted at rest]", str((await plain_session.get_items())[0])[:120], "...")


await demo_live_memory()
# Outside a notebook: asyncio.run(demo_live_memory())

[live sqlite] 4 items stored, turn 2 answer: Your name is Shikamaru Nara.
[live sqlalchemy] 4 items stored, turn 2 answer: Your name is Shikamaru Nara.
[live encrypted] 4 items stored, turn 2 answer: Your name is Shikamaru Nara.
[live encrypted at rest] {'__enc__': 1, 'v': 1, 'kid': 'hkdf-v1', 'payload': 'gAAAAABqw-Wi1nAWV6Zm9Lrry9_sqiezrpJACIs-I8OS5L8XuDZoNBQazX6sBU1xw1D ...


## Squeeze a long chat for real


In [8]:
from typing import (
    Any,  # a type that means any kind of value
)


compaction_decisions: list[str] = []  # a note for each time the SDK asked "should we squeeze now?"


# should_trigger_compaction callback: says yes once there are 2 items to squeeze (the normal rule is 10),
# so the squeeze happens in a short demo
def compact_early(context: dict[str, Any]) -> bool:  # context: the items that could be squeezed, and more
    candidates = len(context["compaction_candidate_items"])
    should_compact = candidates >= 2
    compaction_decisions.append(f"{candidates} candidates, compact: {should_compact}")
    return should_compact

In [9]:
import os  # reads OPENAI_API_KEY from the environment

from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.memory import (
    OpenAIResponsesCompactionSession,  # a session that shrinks old chat history when it gets long
    SQLiteSession,  # a session that saves chat history in a SQLite database (or in memory)
)
from dotenv import load_dotenv  # loads the .env file into the environment

load_dotenv()


# Demo: a real squeeze (compaction) on a three-turn chat, then check the name is still remembered
async def demo_live_compaction() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    # Initialize the agent
    agent = Agent(
        name="Leaf Village Assistant",
        instructions="Answer briefly.",
    )

    # Initialize the session
    underlying = SQLiteSession(
        session_id="ninja-live-compacted",
        db_path=":memory:",
    )
    compaction_session = OpenAIResponsesCompactionSession(
        # session_id: the name of this chat history
        session_id="ninja-live-compacted",
        # underlying_session: where the items really live. Compaction shortens them on top of it
        underlying_session=underlying,
        # model: which model writes the shorter summary
        model="gpt-5.6-luna",
        # should_trigger_compaction: squeeze early so it happens in this short demo
        should_trigger_compaction=compact_early,
    )

    turns = [
        "My name is Shikamaru Nara, remember that.",
        "I'm on Team 10 with Ino and Choji.",
        "What's my name and team?",
    ]
    final_answer = ""
    for number, message in enumerate(turns, start=1):
        # Run the Agent
        result = await Runner.run(
            agent,
            message,
            # session: saves and loads this chat's history
            session=compaction_session,
            # run_config: settings for just this run
            run_config=RunConfig(workflow_name=f"e11-live-compaction-t{number}"),
        )
        final_answer = result.final_output
        print(f"[live compaction turn {number}] saved items: {len(await underlying.get_items())}")
    # proves the SDK asked should_trigger_compaction after each turn and a real squeeze replaced the history
    print("[live compaction decisions]", compaction_decisions)
    # proves the shorter history still carries turn 1's facts into the last turn
    print("[live compaction final answer]", final_answer)


await demo_live_compaction()
# Outside a notebook: asyncio.run(demo_live_compaction())

[live compaction turn 1] saved items: 2
[live compaction turn 2] saved items: 3
[live compaction turn 3] saved items: 5
[live compaction decisions] ['1 candidates, compact: False', '2 candidates, compact: True', '1 candidates, compact: False']
[live compaction final answer] Your name is Shikamaru Nara, and you’re on Team 10 with Ino and Choji.


## Summary

- `session=` on `Runner.run` loads and saves the chat for you. `SQLiteSession` is the quick
  choice, and `SQLAlchemySession` uses any real database in the same way.
- `EncryptedSession` wraps any session so the saved items are scrambled, while `get_items()` still
  gives back normal text. `OpenAIResponsesCompactionSession` shrinks a long chat and needs a live key to run.
